# CIFAR-10 complete project
Select Runtime > Change runtime type > GPU, then run cells in order. All project code is embedded. New timestamped Drive folders preserve previous runs. Default tuning is 20 epochs per trial, three trials per model. Final test results are generated only after training. To evaluate existing checkpoints, set TUNING to their folder and skip the training cell.

In [ ]:
%pip install torch torchvision numpy scikit-learn matplotlib reportlab

In [ ]:
%%writefile CIFAR10_All_In_One.py
# COMPLETE SINGLE-FILE PROJECT
# Install: pip install torch torchvision numpy scikit-learn matplotlib reportlab
# Check: python CIFAR10_All_In_One.py smoke
# Train: python CIFAR10_All_In_One.py tune --epochs 20 --out runs/tuning_new
# Evaluate: python CIFAR10_All_In_One.py finalize --tuning runs/tuning_new --out runs/final_new
# Use fresh output directories. No additional project source files are needed.

#!/usr/bin/env python3
"""CIFAR-10 interim project: complete standalone implementation through 8 Oct 2026.

SETUP (Python 3.11/3.12; GPU recommended for training):
  python -m pip install torch torchvision numpy scikit-learn
RUN:
  python CIFAR10_Complete_Code.py smoke
  python CIFAR10_Complete_Code.py prepare --root data
  python CIFAR10_Complete_Code.py preliminary --out runs/preliminary
  python CIFAR10_Complete_Code.py tune --model all --epochs 20 --out runs/tuning
  python CIFAR10_Complete_Code.py summarize --out runs/tuning

The smoke test uses synthetic inputs, not CIFAR-10 performance data.
Training downloads CIFAR-10 on first use; validation only is used for selection.
The held-out test split is reserved for the 22 October final comparison.
Use fresh run directories: existing results are never overwritten.
No pretrained weights or real experiment outputs are included in this file.
All functions and model definitions are contained here; no sibling Python files
are required. Dependencies must be installed separately. CPU execution works
but the complete 12-trial tuning search can take substantial time.
"""

# models.py
"""Four CIFAR-10 classifiers. Inputs: N x 3 x 32 x 32; outputs: N x 10 logits."""
import torch
from torch import nn

class MLP(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        layers = [nn.Flatten()]
        for a, b in [(3072, 1024), (1024, 512), (512, 256)]:
            layers += [nn.Linear(a, b), nn.BatchNorm1d(b), nn.ReLU(), nn.Dropout(dropout)]
        self.net = nn.Sequential(*layers, nn.Linear(256, 10))
    def forward(self, x):
        return self.net(x)

class ResidualBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(cin, cout, 3, stride, 1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(), nn.Conv2d(cout, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout))
        self.skip = nn.Identity() if cin == cout and stride == 1 else nn.Sequential(
            nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        return torch.relu(self.body(x) + self.skip(x))

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(3, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(),
            ResidualBlock(64,64), ResidualBlock(64,64), ResidualBlock(64,128,2), ResidualBlock(128,128),
            ResidualBlock(128,256,2), ResidualBlock(256,256), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(256,10))
    def forward(self, x):
        return self.net(x)

class RowLSTM(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(96, 256, num_layers=2, dropout=dropout, batch_first=True)
        self.head = nn.Linear(256,10)
    def forward(self, x):
        rows = x.permute(0,2,3,1).reshape(x.size(0),32,96)
        _, (hidden, _) = self.lstm(rows)
        return self.head(hidden[-1])

class ViT(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.patch = nn.Conv2d(3,192,4,4)
        self.cls = nn.Parameter(torch.zeros(1,1,192))
        self.pos = nn.Parameter(torch.zeros(1,65,192))
        layer = nn.TransformerEncoderLayer(192,3,768,dropout,activation='gelu',batch_first=True,norm_first=True)
        self.encoder = nn.TransformerEncoder(layer,6,enable_nested_tensor=False)
        self.norm = nn.LayerNorm(192)
        self.head = nn.Linear(192,10)
        nn.init.trunc_normal_(self.pos,std=.02)
        nn.init.trunc_normal_(self.cls,std=.02)
        # TransformerEncoder clones layers; initialise each clone independently.
        for block in self.encoder.layers:
            nn.init.xavier_uniform_(block.self_attn.in_proj_weight)
            for m in block.modules():
                if isinstance(m,nn.Linear):
                    nn.init.xavier_uniform_(m.weight)
                    if m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self,x):
        x = self.patch(x).flatten(2).transpose(1,2)
        x = torch.cat([self.cls.expand(x.size(0),-1,-1),x],dim=1) + self.pos
        return self.head(self.norm(self.encoder(x)[:,0]))

def build_model(name):
    return {'mlp':MLP,'cnn':CNN,'lstm':RowLSTM,'vit':ViT}[name]()


# data.py
"""Shared stratified split, persisted indices and train-only augmentation."""
import hashlib
import json
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

MEAN=(.4914,.4822,.4465)
STD=(.2470,.2435,.2616)

def get_loaders(root='data', batch_size=128, seed=42):
    if batch_size < 2 or 45000 % batch_size == 1:
        raise ValueError('Choose batch_size >= 2 with 45000 % batch_size != 1 (default 128).')
    root=Path(root); root.mkdir(parents=True,exist_ok=True)
    plain=transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
    aug=transforms.Compose([transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip(),plain])
    training=datasets.CIFAR10(str(root),train=True,download=True,transform=aug)
    validation=datasets.CIFAR10(str(root),train=True,download=False,transform=plain)
    labels=np.asarray(training.targets)
    assert len(labels)==50000 and np.array_equal(np.bincount(labels),np.full(10,5000))
    path=root/f'split_seed_{seed}.json'
    tr,va=train_test_split(np.arange(50000),test_size=5000,stratify=labels,random_state=seed)
    expected={'seed':seed,'train':tr.tolist(),'validation':va.tolist()}
    if path.exists():
        if json.loads(path.read_text()) != expected: raise ValueError('Stored split does not match protocol')
    else: path.write_text(json.dumps(expected))
    assert not set(tr)&set(va)
    assert np.array_equal(np.bincount(labels[va]),np.full(10,500))
    audit={'train_count':len(tr),'validation_count':len(va),'train_class_counts':np.bincount(labels[tr]).tolist(),
           'validation_class_counts':np.bincount(labels[va]).tolist(),'split_sha256':hashlib.sha256(path.read_bytes()).hexdigest()}
    (root/'dataset_audit.json').write_text(json.dumps(audit,indent=2))
    g=torch.Generator().manual_seed(seed)
    # num_workers=0 keeps transform randomness reproducible without worker seed handling.
    return (DataLoader(Subset(training,tr),batch_size,shuffle=True,generator=g,num_workers=0),
            DataLoader(Subset(validation,va),batch_size,shuffle=False,num_workers=0),audit)


# train.py
"""Train and tune on validation data only. Never loads the official test set."""
import argparse
import csv
import json
import math
import random
import time
from pathlib import Path
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

DEFAULT_LR={'mlp':1e-3,'cnn':.1,'lstm':1e-3,'vit':3e-4}

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

def pass_epoch(model,loader,device,optimizer=None):
    model.train(optimizer is not None)
    total=0.; truth=[]; pred=[]
    for x,y in loader:
        x,y=x.to(device),y.to(device)
        with torch.set_grad_enabled(optimizer is not None):
            logits=model(x); loss=torch.nn.functional.cross_entropy(logits,y)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                optimizer.step()
        total+=loss.item()*y.size(0)
        truth.extend(y.detach().cpu().tolist()); pred.extend(logits.argmax(1).detach().cpu().tolist())
    p,r,f,_=precision_recall_fscore_support(truth,pred,labels=list(range(10)),average='macro',zero_division=0)
    return {'loss':total/len(truth),'accuracy':accuracy_score(truth,pred),'macro_precision':float(p),
            'macro_recall':float(r),'macro_f1':float(f),'confusion_matrix':confusion_matrix(truth,pred,labels=list(range(10))).tolist()}

def run(model_name,root,out,epochs=20,lr=None,seed=42,batch_size=128,device='auto',weight_decay=1e-4):
    if epochs<1 or batch_size<2: raise ValueError('epochs >= 1 and batch_size >= 2 required')
    out=Path(out)
    if out.exists() and any(out.iterdir()): raise FileExistsError(f'Use a fresh output directory: {out}')
    out.mkdir(parents=True,exist_ok=True)
    seed_all(seed)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu') if device=='auto' else torch.device(device)
    train,val,audit=get_loaders(root,batch_size,seed)
    model=build_model(model_name).to(device)
    lr=DEFAULT_LR[model_name] if lr is None else lr
    opt=(torch.optim.SGD(model.parameters(),lr=lr,momentum=.9,weight_decay=weight_decay) if model_name=='cnn'
         else torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=weight_decay))
    warmup=min(5,max(1,epochs//5)) if model_name=='vit' else 0
    def multiplier(epoch):
        if epoch<warmup: return (epoch+1)/warmup
        return .5*(1+math.cos(math.pi*(epoch-warmup)/max(1,epochs-warmup)))
    scheduler=torch.optim.lr_scheduler.LambdaLR(opt,multiplier)
    config={'model':model_name,'epochs':epochs,'lr':lr,'seed':seed,'batch_size':batch_size,'weight_decay':weight_decay,
            'device':str(device),'torch_version':torch.__version__,'parameters':sum(p.numel() for p in model.parameters()),
            'selection_metric':'validation macro_f1','data':audit}
    (out/'config.json').write_text(json.dumps(config,indent=2))
    best=-1.; start=time.perf_counter()
    with (out/'history.csv').open('w',newline='') as file:
        writer=csv.DictWriter(file,fieldnames=['epoch','lr','train_loss','train_accuracy','val_loss','val_accuracy','val_macro_f1'])
        writer.writeheader()
        for epoch in range(epochs):
            t=pass_epoch(model,train,device,opt); v=pass_epoch(model,val,device)
            row={'epoch':epoch+1,'lr':opt.param_groups[0]['lr'],'train_loss':t['loss'],'train_accuracy':t['accuracy'],
                 'val_loss':v['loss'],'val_accuracy':v['accuracy'],'val_macro_f1':v['macro_f1']}
            writer.writerow(row); file.flush(); print(json.dumps(row),flush=True)
            scheduler.step()
            checkpoint={'model':model.state_dict(),'optimizer':opt.state_dict(),'scheduler':scheduler.state_dict(),
                        'epoch':epoch+1,'config':config,'validation':v}
            torch.save(checkpoint,out/'last.pt')
            if v['macro_f1']>best:
                best=v['macro_f1']; torch.save(checkpoint,out/'best.pt')
                (out/'best_validation.json').write_text(json.dumps({'epoch':epoch+1,**v},indent=2))
    (out/'runtime.json').write_text(json.dumps({'training_seconds':time.perf_counter()-start},indent=2))
    return json.loads((out/'best_validation.json').read_text())


"""Synthetic checks only: these are NOT CIFAR-10 performance results."""
import json
import torch

def smoke_test():
    torch.manual_seed(42); torch.set_num_threads(2)
    results=[]
    for name in ['mlp','cnn','lstm','vit']:
        model=build_model(name); x=torch.randn(2,3,32,32); y=torch.tensor([0,9])
        model.train(); logits=model(x)
        assert logits.shape==(2,10) and torch.isfinite(logits).all()
        loss=torch.nn.functional.cross_entropy(logits,y); loss.backward()
        assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
        torch.optim.AdamW(model.parameters(),lr=1e-4).step()
        model.eval()
        with torch.no_grad(): assert torch.isfinite(model(x)).all()
        results.append({'model':name,'parameters':sum(p.numel() for p in model.parameters()),'synthetic_check':'passed'})
    print(json.dumps(results,indent=2))

def tune_models(model='all', root='data', out='runs/tuning', epochs=20, device='auto'):
    names=list(DEFAULT_LR) if model=='all' else [model]
    # Check every target before starting so reruns fail before expensive work.
    for name in names:
        for i in range(3):
            folder=Path(out)/name/f'trial_{i}'
            if folder.exists() and any(folder.iterdir()):
                raise FileExistsError(f'Use a fresh output directory: {folder}')
    for name in names:
        trials=[]
        for i,factor in enumerate([.3,1.,3.]):
            folder=Path(out)/name/f'trial_{i}'
            lr=DEFAULT_LR[name]*factor
            metric=run(name,root,folder,epochs=epochs,lr=lr,device=device)
            trials.append({'trial':i,'lr':lr,'checkpoint':str(folder/'best.pt'),**metric})
            (Path(out)/name/'trials.json').write_text(json.dumps(trials,indent=2))
        selected=max(trials,key=lambda item:item['macro_f1'])
        (Path(out)/name/'selected.json').write_text(json.dumps(selected,indent=2))
    summarize(out)


def summarize(out='runs/tuning'):
    """Export actual selected validation results, including explicit pending rows."""
    fields=['model','status','trial','lr','epoch','accuracy','macro_precision','macro_recall','macro_f1','parameters','training_seconds']
    rows=[]
    for name in DEFAULT_LR:
        path=Path(out)/name/'selected.json'
        row={'model':name,'status':'pending'}
        if path.exists():
            result=json.loads(path.read_text())
            row.update({key:result[key] for key in fields if key in result})
            row['status']='completed'
            folder=Path(out)/name/f"trial_{result['trial']}"
            row['parameters']=json.loads((folder/'config.json').read_text())['parameters']
            row['training_seconds']=json.loads((folder/'runtime.json').read_text())['training_seconds']
        rows.append(row)
    target=Path(out);target.mkdir(parents=True,exist_ok=True)
    with (target/'validation_summary.csv').open('w',newline='') as file:
        writer=csv.DictWriter(file,fieldnames=fields);writer.writeheader();writer.writerows(rows)
    print(json.dumps(rows,indent=2))
    print('Scores are fractions, not percentages. These are validation selection results, not final test results.')
    return rows


def main():
    parser=argparse.ArgumentParser(description=__doc__,formatter_class=argparse.RawDescriptionHelpFormatter)
    sub=parser.add_subparsers(dest='command',required=True)
    sub.add_parser('smoke',help='Synthetic forward/backward and optimizer tests; no dataset download')
    prep=sub.add_parser('prepare',help='Download, stratify and audit the training archive')
    prep.add_argument('--root',default='data')
    prelim=sub.add_parser('preliminary',help='Run preliminary MLP and CNN training')
    prelim.add_argument('--root',default='data');prelim.add_argument('--out',default='runs/preliminary')
    prelim.add_argument('--epochs',type=int,default=5);prelim.add_argument('--device',default='auto')
    train=sub.add_parser('train',help='Train one chosen model')
    train.add_argument('--model',choices=list(DEFAULT_LR),required=True)
    train.add_argument('--root',default='data');train.add_argument('--out',required=True)
    train.add_argument('--epochs',type=int,default=20);train.add_argument('--lr',type=float)
    train.add_argument('--seed',type=int,default=42);train.add_argument('--batch-size',type=int,default=128)
    train.add_argument('--weight-decay',type=float,default=1e-4);train.add_argument('--device',default='auto')
    tune=sub.add_parser('tune',help='Three learning rates per model; fixed common epoch budget')
    tune.add_argument('--model',choices=[*DEFAULT_LR,'all'],default='all')
    tune.add_argument('--root',default='data');tune.add_argument('--out',default='runs/tuning')
    tune.add_argument('--epochs',type=int,default=20);tune.add_argument('--device',default='auto')
    summary=sub.add_parser('summarize',help='Export saved selected validation results')
    summary.add_argument('--out',default='runs/tuning')
    args=vars(parser.parse_args());command=args.pop('command')
    if command=='smoke': smoke_test()
    elif command=='prepare':
        seed_all(42);_,_,audit=get_loaders(args['root']);print(json.dumps(audit,indent=2))
    elif command=='preliminary':
        for name in ['mlp','cnn']:
            run(name,args['root'],Path(args['out'])/name,epochs=args['epochs'],device=args['device'])
    elif command=='train':
        args['model_name']=args.pop('model');run(**args)
    elif command=='tune': tune_models(**args)
    elif command=='summarize': summarize(**args)


"""Generate a report from measured results only."""
import json
from pathlib import Path
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, Image
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

def create_report(path,rows,split='validation',artifacts=None):
    styles=getSampleStyleSheet();story=[]
    def p(t): story.append(Paragraph(t,styles['BodyText']));story.append(Spacer(1,10))
    def h(t): story.append(Paragraph(t,styles['Heading2']))
    story.append(Paragraph('Comparative Analysis of Deep Learning Architectures for CIFAR-10',styles['Title']))
    p('ICT-4442 | MLP, residual CNN, row-wise LSTM and Vision Transformer')
    p('Harsh Kushwaha (230953366), Sanket Tippimani (230911200), Sanyam Satija (230911204), Devansh Sharma (230911382)')
    p('Evidence status: '+('held-out test evaluation generated from selected checkpoints.' if split=='test' else 'completed five-epoch validation experiments; final test evaluation pending. This is a results-backed report draft, not a completed test-study submission.'))
    h('1. Abstract and objective')
    p('This project compares four architecture families under a shared CIFAR-10 data protocol. The objective is to examine how local spatial structure, recurrence and attention relate to classification performance. Accuracy, macro precision, recall and F1 are measured together with parameter count and training duration. Conclusions apply to this implementation and training budget; they do not establish universal rankings.')
    h('2. Dataset and experimental protocol')
    p('The official 50,000 training images are stratified into 45,000 training and 5,000 validation images with seed 42. The 10,000 official test images are excluded from tuning. Training uses random crop with padding 4 and horizontal flip. All splits use channel means (0.4914, 0.4822, 0.4465) and standard deviations (0.2470, 0.2435, 0.2616). Labels are integer class indices. Batch size is 128.')
    p('Each architecture has three learning-rate trials at 0.3, 1 and 3 times its base rate. Base rates are MLP 0.001, CNN 0.1, LSTM 0.001 and ViT 0.0003. Selection uses the highest validation macro-F1, first within each run and then across trials. The supplied completed experiment uses five epochs per trial; the planned twenty-epoch protocol requires a new run. Final evaluation freezes this selection before loading the test set.')
    h('3. Models and optimisation')
    p('MLP: flattened 3,072 features, dense widths 1,024/512/256, BatchNorm, ReLU and dropout 0.3. CNN: 64-channel stem, six residual blocks across 64/128/256 channels, global average pooling and linear head. LSTM: 32 image rows of 96 features, two recurrent layers of width 256 and a final hidden-state classifier. ViT: 4x4 patch projection, 64 patch tokens plus CLS, dimension 192, six encoder layers, three attention heads and feed-forward dimension 768.')
    p('Cross-entropy, gradient clipping at 1.0 and weight decay 0.0001 are shared. CNN uses SGD with momentum 0.9; others use AdamW. Learning rates follow cosine decay with ViT warm-up. This implementation does not apply label smoothing, Cutout, pretraining or early stopping. Equal epochs and trial counts do not imply equal compute or optimisers.')
    story.append(PageBreak());h('4. Measured '+split+' results')
    data=[['Model','Accuracy','Precision','Recall','Macro-F1','Params (M)']]
    for r in rows:data.append([r['model'].upper(),f"{r['accuracy']*100:.2f}%",f"{r['macro_precision']:.4f}",f"{r['macro_recall']:.4f}",f"{r['macro_f1']:.4f}",f"{r['parameters']/1e6:.3f}"])
    t=Table(data,colWidths=[65,75,75,65,75,85]);t.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#173850')),('TEXTCOLOR',(0,0),(-1,0),colors.white),('FONTSIZE',(0,0),(-1,-1),9),('BOTTOMPADDING',(0,0),(-1,-1),10),('GRID',(0,0),(-1,-1),.3,colors.lightgrey)]));story.append(t);story.append(Spacer(1,12))
    for r in rows:p(f"{r['model'].upper()}: selected trial training time {r['training_seconds']/60:.2f} minutes. This is not the total tuning cost.")
    best=max(rows,key=lambda r:r['accuracy']);p(f"The highest observed {split} accuracy is {best['accuracy']*100:.2f}% for {best['model'].upper()}. The implemented ViT has 2.694M parameters; the interim report's 1.807M figure does not describe this implementation.")
    h('5. Discussion and limitations')
    p('The observed CNN advantage is consistent with its local receptive fields and shared convolution weights. The MLP lacks explicit spatial structure; the LSTM compresses row sequences into a final hidden state; the ViT learns patch interactions with fewer built-in spatial constraints. These mechanisms provide plausible explanations, not causal proof. Five epochs may penalise slower-converging models. A single seed cannot quantify training variability. Validation selection is optimistic relative to untouched test evaluation. Historical 15-epoch MLP/CNN numbers use a different budget and must not be merged with this comparison.')
    h('6. Error analysis and reproducibility')
    p('The final evaluation exports class precision/recall/F1, confusion counts, every prediction with confidence, the five largest directed confusion pairs, learning curves and twelve highest-confidence mistakes per model. Example images require human inspection before attributing errors to background, pose or visual similarity. Checkpoint SHA-256 hashes and training configuration accompany each test result. No specific confusion pairs are asserted from the uploaded notebook, because predictions and weights were not attached.')
    h('7. Completion and contribution record')
    p('Code coverage includes all four models, training, learning-rate search, test evaluation, figures and automated reporting. Actual remaining actions are to run the selected protocol on Colab, inspect the generated errors, verify member contributions and complete institutional submission checks. Proposed ownership follows the synopsis: Harsh - MLP; Sanket - CNN; Sanyam - LSTM; Devansh - ViT. Ownership is not proof of completed commits or contributions.')
    h('8. Source documents')
    p('Project scope and architecture rationale: supplied Deep Learning Synopsis and ICT-4442 Interim Report. Measured validation evidence: saved outputs in CIFAR10_Complete_Code (1).ipynb. This report does not independently verify literature benchmark claims from those documents.')
    if artifacts:
        for r in rows:
            story.append(PageBreak());h(r['model'].upper()+' - test analysis')
            for f in ['confusion_matrix.png','errors.png']:
                story.append(Image(str(Path(artifacts)/r['model']/f),width=450,height=350))
    def footer(c,d):c.drawString(45,25,f'ICT-4442 | {split} evidence | Page {d.page}')
    SimpleDocTemplate(str(path),rightMargin=45,leftMargin=45,topMargin=40,bottomMargin=42).build(story,onFirstPage=footer,onLaterPages=footer)

"""Evaluate frozen validation-selected checkpoints and generate actual test artifacts."""
import argparse, csv, hashlib, json
from pathlib import Path
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import sys
core = sys.modules[__name__]

def evaluate(tuning, root, out):
    tuning, out = Path(tuning), Path(out)
    if out.exists():
        raise FileExistsError(f'Refusing to overwrite final evaluation: {out}')
    selections=[]
    for name in core.DEFAULT_LR:
        selected=json.loads((tuning/name/'selected.json').read_text())
        checkpoint=tuning/name/f"trial_{selected['trial']}"/'best.pt'
        if not checkpoint.exists(): raise FileNotFoundError(checkpoint)
        selections.append((name,checkpoint))
    out.mkdir(parents=True)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    transform=core.transforms.Compose([core.transforms.ToTensor(),core.transforms.Normalize(core.MEAN,core.STD)])
    dataset=core.datasets.CIFAR10(root,train=False,download=True,transform=transform)
    loader=core.DataLoader(dataset,128,shuffle=False,num_workers=0)
    rows=[]
    for name,path in selections:
        folder=out/name; folder.mkdir()
        # Only load checkpoints produced by this project and trusted by the user.
        ck=torch.load(path,map_location='cpu',weights_only=False)
        model=core.build_model(name).to(device);model.load_state_dict(ck['model']);model.eval()
        truth=[];pred=[];confidence=[]
        with torch.inference_mode():
            for x,y in loader:
                probs=model(x.to(device)).softmax(1)
                score,label=probs.max(1)
                truth.extend(y.tolist());pred.extend(label.cpu().tolist());confidence.extend(score.cpu().tolist())
        report=classification_report(truth,pred,labels=list(range(10)),target_names=dataset.classes,output_dict=True,zero_division=0)
        cm=confusion_matrix(truth,pred,labels=list(range(10)))
        (folder/'classification_report.json').write_text(json.dumps(report,indent=2))
        np.savetxt(folder/'confusion_matrix.csv',cm,delimiter=',',fmt='%d')
        with (folder/'predictions.csv').open('w',newline='') as f:
            w=csv.writer(f);w.writerow(['test_index','true_class','predicted_class','confidence']);w.writerows(zip(range(len(truth)),truth,pred,confidence))
        fig,ax=plt.subplots(figsize=(9,8));im=ax.imshow(cm,cmap='Blues');fig.colorbar(im,ax=ax)
        ax.set(xticks=range(10),yticks=range(10),xticklabels=dataset.classes,yticklabels=dataset.classes,xlabel='Predicted',ylabel='True',title=f'{name.upper()} - held-out test counts')
        plt.setp(ax.get_xticklabels(),rotation=45,ha='right')
        for i in range(10):
            for j in range(10): ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black',fontsize=8)
        fig.tight_layout();fig.savefig(folder/'confusion_matrix.png',dpi=160);plt.close(fig)
        wrong=np.flatnonzero(np.asarray(truth)!=np.asarray(pred)); wrong=sorted(wrong,key=lambda i:-confidence[i])[:12]
        fig,axs=plt.subplots(3,4,figsize=(12,9))
        for ax in axs.flat: ax.axis('off')
        for ax,i in zip(axs.flat,wrong):
            ax.imshow(dataset.data[i]);ax.set_title(f'#{i} True: {dataset.classes[truth[i]]}\nPred: {dataset.classes[pred[i]]} ({confidence[i]:.1%})',fontsize=9)
        fig.suptitle(f'{name.upper()}: highest-confidence errors (selected examples)');fig.tight_layout();fig.savefig(folder/'errors.png',dpi=160);plt.close(fig)
        pairs=sorted([(int(cm[i,j]),dataset.classes[i],dataset.classes[j]) for i in range(10) for j in range(10) if i!=j],reverse=True)[:5]
        (folder/'error_analysis.txt').write_text('\n'.join(f'{a} -> {b}: {n}' for n,a,b in pairs)+'\nVisual examples are confidence-selected, not a representative random sample. Inspect images before assigning causes.')
        config=ck['config']; runtime=json.loads((path.parent/'runtime.json').read_text())
        rows.append(dict(model=name,accuracy=report['accuracy'],macro_precision=report['macro avg']['precision'],macro_recall=report['macro avg']['recall'],macro_f1=report['macro avg']['f1-score'],parameters=config['parameters'],training_seconds=runtime['training_seconds'],epochs=config['epochs']))
        (folder/'provenance.json').write_text(json.dumps({'checkpoint':str(path),'sha256':hashlib.sha256(path.read_bytes()).hexdigest(),'config':config,'selected_epoch':ck['epoch'],'split':'official 10000-image test set'},indent=2))
        history=list(csv.DictReader((path.parent/'history.csv').open()))
        fig,axes=plt.subplots(1,2,figsize=(11,4))
        for ax,keys in zip(axes,[['train_loss','val_loss'],['train_accuracy','val_accuracy']]):
            for key in keys: ax.plot([int(r['epoch']) for r in history],[float(r[key]) for r in history],label=key)
            ax.set_xlabel('Epoch');ax.legend()
        fig.tight_layout();fig.savefig(folder/'learning_curves.png',dpi=160);plt.close(fig)
    with (out/'test_comparison.csv').open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
    (out/'test_comparison.json').write_text(json.dumps(rows,indent=2))
    fig,ax=plt.subplots(figsize=(8,5));ax.bar([r['model'].upper() for r in rows],[100*r['accuracy'] for r in rows]);ax.set(ylabel='Test accuracy (%)',ylim=(0,100));fig.tight_layout();fig.savefig(out/'test_comparison.png',dpi=160);plt.close(fig)
    create_report(out/'Final_Report.pdf', rows, 'test', out)
    print(f'Completed evaluation and report: {out}')


if __name__ == '__main__':
    import sys
    if len(sys.argv) > 1 and sys.argv[1] == 'finalize':
        parser = argparse.ArgumentParser(description='Evaluate selected checkpoints and create final report')
        parser.add_argument('--tuning', required=True)
        parser.add_argument('--root', default='data')
        parser.add_argument('--out', required=True)
        args = parser.parse_args(sys.argv[2:])
        evaluate(args.tuning, args.root, args.out)
    else:
        main()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from datetime import datetime
from pathlib import Path
import subprocess, sys
BASE=Path('/content/drive/MyDrive/CIFAR10_Final')/datetime.now().strftime('%Y%m%d_%H%M%S')
TUNING=BASE/'tuning'
FINAL=BASE/'final'
EPOCHS=20
def run(*args):
    subprocess.run([sys.executable,'-u','CIFAR10_All_In_One.py',*map(str,args)],check=True)
print('Results:',BASE)
run('smoke')

In [ ]:
# Skip only if TUNING points to a complete existing tuning run.
run('tune','--epochs',EPOCHS,'--out',TUNING)

In [ ]:
run('finalize','--tuning',TUNING,'--out',FINAL)

In [ ]:
from google.colab import files
files.download(str(FINAL/'Final_Report.pdf'))